# Bibliography Extraction

Parse messy OCR/HTML bibliography into structured data using LLMs.

In [1]:
from largeliterarymodels.tasks.extract_bibliography import BibliographyTask, BibliographyEntry, chunk_bibliography
from largeliterarymodels.llm import GEMINI_FLASH

MODEL = GEMINI_FLASH

task = BibliographyTask()
task

BibliographyTask(name='BibliographyTask', schema=list[BibliographyEntry])

In [3]:
# task.stash.clear()

## Load and slice the bibliography

Replace the path below with your OCR/HTML file. Then split into chunks — one chunk per year heading is a natural boundary.

In [4]:
# Load your file
biblio_path = "../data/English prose fiction, 1600-1700 -- a chronological checklist.html"
with open(biblio_path) as f:
    raw_html = f.read()

chunks = chunk_bibliography(raw_html, max_entries=10)
len(chunks)

196

In [5]:
chunk = [c for c in chunks if "PALMERIN D’OLIVA" in c][0]
print(chunk)

<h2 style="font-weight:normal;"><a id="bookmark17"></a><span class="font5">1615</span></h2>
<p><span class="font5">FORDE, EMANUEL. Parismus, the renoumed Prince of Bohemia. T. Crecde. (STC 11173. Third edition.)</span></p>
<p><span class="font5">GREENE, ROBERT. Philomela, the Lady Fitzwaters nightingale. G. Purslowe. (STC 12297. Second edition; first edition in 1592.)</span></p>
<p><span class="font5">JACK OF DOVER. Jacke of Dovers merry' tales. Or his quest of inquiry'. J. B., sold by R. Higgenbotham. (STC 14292. Third edition.)</span></p>
<p><span class="font5">PALMERIN D’OLIVA. Palmerin d’Oliva: turned into English by A[nthony] M[unday], T. Creede. (STC 19159. Third edition; earlier editions in 1588, 1597.)</span></p>


## Run extraction

Run a single chunk first to verify, then batch all chunks.

In [6]:
# Test on one chunk
entries = task.run(chunk, model=MODEL)
for e in entries:
    print(f"  {e.year} | {e.author} | {e.title} | printer={e.printer} | pub={e.publisher}")


  1615 | Forde, Emanuel | Parismus | printer=T. Creede | pub=
  1615 | Greene, Robert | Philomela | printer=G. Purslowe | pub=
  1615 |  | Jack of Dover | printer=J. B. | pub=
  1615 |  | Palmerin d’Oliva | printer=T. Creede | pub=


In [7]:
# Run all chunks (cached — safe to re-run)
all_entries = task.map(chunks, model=MODEL)
len(all_entries)

Extracting list[BibliographyEntry] (gemini-2.5-flash):   0%|          | 0/195 [00:00<?, ?it/s]

Extracting list[BibliographyEntry] (gemini-2.5-flash): 100%|██████████| 195/195 [16:37<00:00,  5.12s/it]


196

## Export to DataFrame / CSV

In [8]:
df = task.df
df = df.sort_values(["year", "author"]).reset_index(drop=True)
print(f"{len(df)} rows, {len(df.columns)} columns")
df.head(10)

1456 rows, 18 columns


,model,temperature,prompt,author,title,title_sub,year,edition,id_biblio,is_translated,translated_from,translator,printer,publisher,bookseller,notes_biblio,notes,uncertain_author
0,gemini-2.5-flash,0.7,"<h2 style=""font-weight:normal;""><a id=""bookmar...",,Albions Queen,. The famous history of Albions queene.,1600,,STC 11502,False,,,W. White,T. Pavier,,Dedication signed R. G.,,False
1,gemini-2.5-flash,0.7,"<h2 style=""font-weight:normal;""><a id=""bookmar...",,Gesta Romanorum,". A record of auncient histories, intituled in...",1600,Sixth impression,,True,Latin,,T. East,,,Esdaile; Boston Public Library. Sixth edition ...,,False
2,gemini-2.5-flash,0.7,"<h2 style=""font-weight:normal;""><a id=""bookmar...",,Oceander,. The heroicall adventures of the Knight of th...,1600,,STC 18763,False,,,,W. Leake,,,,False
3,gemini-2.5-flash,0.7,"<h2 style=""font-weight:normal;""><a id=""bookmar...","Armin, Robert",Foole upon foole,", or Six sortes of sottes. [By] Clonnico de Cu...",1600,First edition,,False,,,,W. Ferbrand,,Folger Library. Reprinted 1605; re-issued with...,,False
4,gemini-2.5-flash,0.7,"<h2 style=""font-weight:normal;""><a id=""bookmar...","Breton, Nicholas",The strange fortunes of two excellent princes,,1600,,STC 3702,False,,,P. Short,N. Ling,,,,False
5,gemini-2.5-flash,0.7,"<h2 style=""font-weight:normal;""><a id=""bookmar...","Chambers, Robert",Palestina,written by Mr. R. C. P. and Bachelor of Divini...,1600,,STC 4954,False,,,,B. Sermartelli [i.e. J. Wolfe?],,Florence [i.e. London?]:,,False
6,gemini-2.5-flash,0.7,﻿,"Greene, Robert",Greenes Never too late,,1600,Second edition,STC 12254,False,,,J. Roberts,N. Ling,,First edition in 1590.,,False
7,gemini-2.5-flash,0.7,"<h2 style=""font-weight:normal;""><a id=""bookmar...","Greene, Robert",Greenes Groats-worth of wit,,1600,Third edition,,False,,,,,,"Cited by Esdaile, without imprint, from the Ce...",,False
8,gemini-2.5-flash,0.7,"<h2 style=""font-weight:normal;""><a id=""bookmar...","Greene, Robert",Greenes Never too late,,1600,Second edition,STC 12254,False,,,J. Roberts,N. Ling,,First edition in 1590.,,False
9,gemini-2.5-flash,0.7,"<h2 style=""font-weight:normal;""><a id=""bookmar...","Kittowe, Robert",Loves load-starre,,1600,,STC 15026,False,,,T. Creede,,,,,False


In [10]:
# Save to CSV
output_path = "../data/bibliography.csv"
df.drop(columns=["prompt","model","temperature"]).to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Saved to ../data/bibliography.csv


## Compare models (optional)

Run the same extraction with a different model to compare results.

In [ ]:
# Compare on a single chunk
for model in ["claude-sonnet-4-20250514", "gpt-4o-mini", "gemini-2.5-flash"]:
    entries = task.run(chunks[0], model=model)
    print(f"\n{model}: {len(entries)} entries")
    for e in entries[:2]:
        print(f"  {e.year} | {e.author} | {e.title}")